In [ ]:
import os
import pandas as pd
from typing import Optional, Tuple
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
import yfinance as yf

# Prefer a repo-relative default; allow override via env var.
input_path = os.getenv("MODEL_INPUT_PATH", "model_input.csv")
df = pd.read_csv(input_path)

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
train_mask = df["date"] < "2026-01-01"

In [ ]:
def pick(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    norm = lambda s: s.strip().lower().replace(" ", "_").replace("-", "_")
    norm_map = {norm(c): c for c in df.columns}
    for c in candidates:
        if norm(c) in norm_map:
            return norm_map[norm(c)]
    return None


def clean_unq_key(x) -> str:
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


PRUNE_CFG = {
    "drop_cat_zero_month": True,      # drop months where cat_actual_m == 0
    "drop_channel_zero_month": True,  # drop months where chan_actual_in_cat_m == 0
    "drop_all_zero_row": True,        # drop if actual_m==0 & cat==0 & channel==0
    "drop_invalid_zero": True,        # drop if invalid_actual_count_m>0 and actual_m==0
    "drop_inactive_months": False,    # if active_sales_m exists: drop if active_sales_m==0 and actual_m==0
}


def monthly_feature_extraction(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    # Resolve column names
    date_col = pick(df, ["date", "tarih"])
    alt_kat_col = pick(df, ["alt_kategori", "alt kategori", "altkategori"])
    urun_adi_col = pick(df, ["urun_adı", "urun_adi", "product_name", "item_name"])
    unq_key_col = pick(df, ["unq_key"])
    urun_kodu_col = pick(df, ["urun_kodu", "product_code", "sku"])
    satis_hacmi_col = pick(df, ["satis_hacmi", "satış_hacmi", "satis hacmi", "channel", "kanal"])
    actual_col = pick(df, ["actual", "satis", "satış", "sales", "sales_qty", "qty", "quantity"])
    active_col = pick(df, ["active_sales", "aktif_satis", "aktif_satış", "is_active", "active"])

    required = {
        "date": date_col,
        "alt_kategori": alt_kat_col,
        "urun_adı": urun_adi_col,
        "unq_key": unq_key_col,
        "urun_kodu_col": urun_kodu_col,
        "satis_hacmi": satis_hacmi_col,
        "actual": actual_col,
    }
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required field(s): {missing}. Please check column names.")

    df = df.copy()

    # Mask to detect non-numeric records in the original actual column
    _actual_num = pd.to_numeric(df[actual_col], errors="coerce")
    df["_actual_invalid"] = _actual_num.isna() & df[actual_col].notna()

    # Type casting / cleanup
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df[alt_kat_col] = df[alt_kat_col].astype(str).str.strip()
    df[urun_adi_col] = df[urun_adi_col].astype(str).str.strip()
    df[unq_key_col] = df[unq_key_col].map(clean_unq_key)
    df[urun_kodu_col] = df[urun_kodu_col].map(clean_unq_key)
    df[satis_hacmi_col] = df[satis_hacmi_col].astype(str).str.strip()
    df[actual_col] = _actual_num.fillna(0.0)

    if active_col is not None:
        df[active_col] = pd.to_numeric(df[active_col], errors="coerce").fillna(0)
        df[active_col] = (df[active_col] != 0).astype(int)

    # Monthly bucket
    df["month"] = df[date_col].values.astype("datetime64[M]")

    # Keys
    keys = [alt_kat_col, urun_adi_col, unq_key_col, urun_kodu_col, satis_hacmi_col, "month"]

    # Monthly total sales + invalid count + (optional) active_sales_m
    agg_sales = df.groupby(keys, as_index=False)[actual_col].sum().rename(columns={actual_col: "actual_m"})
    agg_invalid = df.groupby(keys, as_index=False)["_actual_invalid"].sum().rename(columns={"_actual_invalid": "invalid_actual_count_m"})
    agg = agg_sales.merge(agg_invalid, on=keys, how="left")
    if active_col is not None:
        agg_active = df.groupby(keys, as_index=False)[active_col].max().rename(columns={active_col: "active_sales_m"})
        agg = agg.merge(agg_active, on=keys, how="left")
    agg = agg.sort_values(keys)

    # -------- Category-month context --------
    cat_month = (
        agg.groupby([alt_kat_col, "month"], as_index=False)["actual_m"]
        .sum().rename(columns={"actual_m": "cat_actual_m"})
    )
    agg = agg.merge(cat_month, on=[alt_kat_col, "month"], how="left")

    agg["ratio_in_cat_m"] = agg["actual_m"] / agg["cat_actual_m"].replace({0: np.nan})

    # Product name share within category-month
    name_tot = (
        agg.groupby([alt_kat_col, urun_adi_col, "month"], as_index=False)["actual_m"]
        .sum().rename(columns={"actual_m": "name_actual_in_cat_m"})
    )
    agg = agg.merge(name_tot, on=[alt_kat_col, urun_adi_col, "month"], how="left")
    agg["name_share_in_cat_m"] = agg["name_actual_in_cat_m"] / agg["cat_actual_m"].replace({0: np.nan})
    agg = agg.drop(columns=["name_actual_in_cat_m"])

    # unq_key total within category-month (rename)
    code_tot = (
        agg.groupby([alt_kat_col, unq_key_col, "month"], as_index=False)["actual_m"]
        .sum().rename(columns={"actual_m": "unq_key_actual_in_cat_m"})
    )
    agg = agg.merge(code_tot, on=[alt_kat_col, unq_key_col, "month"], how="left")

    # -------- Channel within category-month context --------
    chan_tot_in_cat = (
        agg.groupby([alt_kat_col, "month", satis_hacmi_col], as_index=False)["actual_m"]
        .sum().rename(columns={"actual_m": "chan_actual_in_cat_m"})
    )
    agg = agg.merge(chan_tot_in_cat, on=[alt_kat_col, "month", satis_hacmi_col], how="left")
    agg["ratio_in_channel_m"] = agg["actual_m"] / agg["chan_actual_in_cat_m"].replace({0: np.nan})
    chan_weight_in_cat = (
        chan_tot_in_cat.assign(
            chan_weight_in_cat_m=lambda x: x["chan_actual_in_cat_m"] /
            x.groupby([alt_kat_col, "month"])["chan_actual_in_cat_m"].transform("sum").replace({0: np.nan})
        )[[alt_kat_col, "month", satis_hacmi_col, "chan_weight_in_cat_m"]]
    )
    agg = agg.merge(chan_weight_in_cat, on=[alt_kat_col, "month", satis_hacmi_col], how="left")

    # -------- Initial ranking (for positive sales only) --------
    agg["rank_in_cat_m"] = (
        agg.groupby([alt_kat_col, "month"])["actual_m"]
        .rank(method="dense", ascending=False).astype("Float64")
    )
    invalid_rank_mask = (agg["actual_m"] <= 0) | (~np.isfinite(agg["actual_m"])) | (agg["cat_actual_m"] <= 0)
    agg.loc[invalid_rank_mask, "rank_in_cat_m"] = pd.NA
    agg["rank_in_cat_m"] = agg["rank_in_cat_m"].astype("Int64")

    # -------- Lags / rolling / growth for monthly model --------
    group_keys = [alt_kat_col, urun_adi_col, unq_key_col, satis_hacmi_col]
    for lag in (1, 3, 12):
        agg[f"lag_{lag}m"] = agg.groupby(group_keys)["actual_m"].shift(lag)
    for w in (3, 6, 12):
        grp = agg.groupby(group_keys)["actual_m"]
        agg[f"roll_mean_{w}m"] = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    agg["mom_growth"] = (agg["actual_m"] / agg["lag_1m"] - 1).replace([np.inf, -np.inf], np.nan)
    agg["yoy_growth"] = (agg["actual_m"] / agg["lag_12m"] - 1).replace([np.inf, -np.inf], np.nan)

    # -------- Quality flags --------
    agg["flag_cat_zero_month"] = (agg["cat_actual_m"].fillna(0) == 0).astype(int)
    agg["flag_channel_zero_month"] = (agg["chan_actual_in_cat_m"].fillna(0) == 0).astype(int)
    agg["flag_all_zero_row"] = (
        (agg["actual_m"].fillna(0) == 0) &
        (agg["cat_actual_m"].fillna(0) == 0) &
        (agg["chan_actual_in_cat_m"].fillna(0) == 0)
    ).astype(int)
    agg["flag_invalid_zero"] = ((agg["invalid_actual_count_m"] > 0) & (agg["actual_m"] == 0)).astype(int)

    if "active_sales_m" in agg.columns:
        agg["flag_inactive_month"] = ((agg["active_sales_m"] == 0) & (agg["actual_m"] == 0)).astype(int)

    # -------- Optional prune --------
    prune_mask = pd.Series(False, index=agg.index)
    if PRUNE_CFG["drop_cat_zero_month"]:
        prune_mask |= agg["flag_cat_zero_month"] == 1
    if PRUNE_CFG["drop_channel_zero_month"]:
        prune_mask |= agg["flag_channel_zero_month"] == 1
    if PRUNE_CFG["drop_all_zero_row"]:
        prune_mask |= agg["flag_all_zero_row"] == 1
    if PRUNE_CFG["drop_invalid_zero"]:
        prune_mask |= agg["flag_invalid_zero"] == 1
    if PRUNE_CFG.get("drop_inactive_months", False) and "flag_inactive_month" in agg.columns:
        prune_mask |= agg["flag_inactive_month"] == 1

    # Prune
    agg_pruned = agg.loc[~prune_mask].copy()

    # Recompute ranks after pruning (within-group ordering may change)
    agg_pruned["rank_in_cat_m"] = (
        agg_pruned.groupby([alt_kat_col, "month"])["actual_m"]
        .rank(method="dense", ascending=False).astype("Float64")
    )
    invalid_rank_mask2 = (agg_pruned["actual_m"] <= 0) | (~np.isfinite(agg_pruned["actual_m"])) | (agg_pruned["cat_actual_m"] <= 0)
    agg_pruned.loc[invalid_rank_mask2, "rank_in_cat_m"] = pd.NA
    agg_pruned["rank_in_cat_m"] = agg_pruned["rank_in_cat_m"].astype("Int64")

    # Ratios: NaN/inf -> NaN, clip negatives to 0 (keep NaNs)
    for c in [
        "ratio_in_cat_m", "name_share_in_cat_m", "ratio_in_channel_m", "chan_weight_in_cat_m",
        "mom_growth", "yoy_growth",
    ]:
        if c in agg_pruned.columns:
            agg_pruned.loc[~np.isfinite(agg_pruned[c]), c] = np.nan
            agg_pruned[c] = agg_pruned[c].clip(lower=0)

    # Round to 3 decimals (float columns only)
    float_cols = agg_pruned.select_dtypes(include=["float64", "float32", "Float64"]).columns
    agg_pruned[float_cols] = agg_pruned[float_cols].round(3)

    meta = {
        "keys": keys,
        "counts": {
            "rows_in": len(df),
            "rows_out_before_prune": len(agg),
            "rows_out_after_prune": len(agg_pruned),
            "n_alt_kategori": df[alt_kat_col].nunique(dropna=True),
            "n_urun_adi": df[urun_adi_col].nunique(dropna=True),
            "n_unq_key": df[unq_key_col].nunique(dropna=True),
            "n_satis_hacmi": df[satis_hacmi_col].nunique(dropna=True),
            "date_range": (pd.to_datetime(df[date_col]).min(), pd.to_datetime(df[date_col]).max()),
        },
    }
    return agg_pruned, meta


a, b = monthly_feature_extraction(df)

In [ ]:
def lagged_sales(df, key_column, date_column, sales_column, fill_na, max_lag):
    """
    For each unique key in `key_column`, sorts by `date_column`,
    finds the lag (from 1 to `max_lag`) of `sales_column` that has the highest correlation with the current value,
    and creates a new column 'lagged_sales' with that lagged value.
    Optionally fills NaNs in 'lagged_sales' with the current sales value.
    """
    # date_column = "SAS_Date"
    # key_column = "unq_key"
    # sales_column = "ACTUAL"
    # fill_na = True
    # max_lag = 10
    print(f"\nCalculating lagged sales feature for {sales_column} with max lag {max_lag}...")

    if not np.issubdtype(df[date_column].dtype, np.datetime64):
        df[date_column] = pd.to_datetime(df[date_column])
    df = df.sort_values([key_column, date_column])

    correlations = {}
    lagged_sales_col = None
    max_corr = None
    for lag in range(1, max_lag + 1):
        lagged = df.groupby(key_column)[sales_column].shift(lag)
        valid = df[sales_column].copy()
        valid_lagged = lagged.copy()
        mask = (~valid.isna()) & (~valid_lagged.isna())
        if mask.sum() > 0:
            corr = valid[mask].corr(valid_lagged[mask])
            correlations[lag] = corr
        else:
            correlations[lag] = None
    max_lag = max(
        correlations,
        key=lambda k: abs(correlations[k]) if correlations[k] is not None else float("-inf"),
    )
    max_corr = correlations[max_lag]
    df["lagged_sales"] = df.groupby(key_column)[sales_column].shift(max_lag)
    print(f"Highest lag: {max_lag}")
    print(f"Correlation at highest lag: {max_corr}")
    df.head()

    if fill_na:
        df["lagged_sales"] = df["lagged_sales"].fillna(df[sales_column])

    print("\nLagged sales feature added.")
    return df


def pct_change_sales(df, key_column, date_column, sales_column, fill_na, max_lag):
    """
    For each unique key in `key_column`, sorts by `date_column`,
    calculates percent change in `sales_column` over lags 1 to `max_lag`,
    finds the lag with the highest correlation with ACTUAL, and creates a single column 'pct_change_sales' for that lag.
    Optionally fills NaNs in the column with 0.
    """
    # date_column = "SAS_Date"
    # key_column = "unq_key"
    # sales_column = "ACTUAL"
    # fill_na = True
    # max_lag = 10
    print(f"\nCalculating percent change sales feature for {sales_column} with max lag {max_lag}...")

    if not np.issubdtype(df[date_column].dtype, np.datetime64):
        df[date_column] = pd.to_datetime(df[date_column])
    df = df.copy()
    df["smoothed sales"] = df[sales_column].replace(0, 0.000001)  # to avoid division by zero
    df = df.sort_values([key_column, date_column])

    corrs = {}
    best_lag_col = None
    for lag in range(1, max_lag + 1):
        col_name = f"pct_change_sales_lag_{lag}"
        temp_col = df.groupby(key_column)["smoothed sales"].pct_change(periods=lag)
        temp_col = temp_col.fillna(0)
        valid = df[["smoothed sales"]].copy()
        valid["temp_col"] = temp_col
        valid = valid.dropna()
        if not valid.empty:
            corr = valid["smoothed sales"].corr(valid["temp_col"])
            corrs[lag] = corr
        else:
            corrs[lag] = None
        if best_lag_col is None or (
            corrs[lag] is not None and abs(corrs[lag]) > abs(corrs.get(best_lag_col, 0))
        ):
            best_lag_col = lag

    df["pct_change_sales"] = df.groupby(key_column)["smoothed sales"].pct_change(periods=best_lag_col)
    if fill_na:
        df["pct_change_sales"] = df["pct_change_sales"].fillna(0)
    df = df.drop(columns=["smoothed sales"])
    print(f"Highest correlation: {corrs[best_lag_col]} at lag {best_lag_col}")
    print("\nPercent change sales feature added.")
    return df


def rolling_avg_sales(df, key_column, date_column, sales_column, window_size):
    """
    For each unique key in `key_column`, sorts by `date_column`,
    calculates the rolling average of `sales_column` over window sizes 1 to `window_size`,
    finds the window size with the highest correlation with `sales_column`, and creates a single column 'rolling_avg_sales' for that window.
    """
    # date_column = "SAS_Date"
    # key_column = "unq_key"
    # sales_column = "ACTUAL"
    # fill_na = True
    # window_size = 10
    print(f"\nCalculating rolling average sales feature for {sales_column} with max window size {window_size}...")

    if not np.issubdtype(df[date_column].dtype, np.datetime64):
        df[date_column] = pd.to_datetime(df[date_column])
    df = df.copy()
    df = df.sort_values([key_column, date_column])
    corrs = {}
    best_window = None
    for window in range(2, window_size + 1):
        temp_col = df.groupby(key_column)[sales_column].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        valid = df[[sales_column]].copy()
        valid["temp_col"] = temp_col
        valid = valid.dropna()
        if not valid.empty:
            corr = valid[sales_column].corr(valid["temp_col"])
            corrs[window] = corr
        else:
            corrs[window] = None
        if best_window is None or (
            corrs[window] is not None and abs(corrs[window]) > abs(corrs.get(best_window, 0))
        ):
            best_window = window
    df["rolling_avg_sales"] = df.groupby(key_column)[sales_column].transform(
        lambda x: x.rolling(window=best_window, min_periods=2).mean()
    )
    print(f"Highest correlation: {corrs[best_window]} at window size {best_window}")
    print("\nRolling average sales feature added.")
    return df


def add_tryusd(df, date_column, fx_ticker):
    """
    Downloads TRY/USD FX data from Yahoo Finance for the date range in `date_column`,
    merges the FX rate to the DataFrame by date, fills missing FX values with previous/next available value,
    and returns the updated DataFrame with a new column 'TRYUSD'.
    """
    # date_column = "SAS_Date"
    # fx_ticker = 'TRYUSD=X'
    print(f"\nDownloading {fx_ticker} data from Yahoo Finance...")

    if not np.issubdtype(df[date_column].dtype, np.datetime64):
        df[date_column] = pd.to_datetime(df[date_column], errors="coerce")
    unique_dates = df[date_column].dropna().unique()

    if len(unique_dates) > 0:
        start_date = pd.to_datetime(min(unique_dates)).strftime("%Y-%m-%d")
        end_date = pd.to_datetime(max(unique_dates)).strftime("%Y-%m-%d")

        try:
            fx = yf.download(fx_ticker, start=start_date, end=end_date)
            if isinstance(fx.columns, pd.MultiIndex):
                fx.columns = ["_".join(col).strip() for col in fx.columns.values]
            close_col = [col for col in fx.columns if "Close" in col][0]
            fx = fx.reset_index()
            fx["Date"] = pd.to_datetime(fx["Date"]).dt.date
            df["date_only"] = pd.to_datetime(df[date_column]).dt.date
            df = df.merge(fx[["Date", close_col]], left_on="date_only", right_on="Date", how="left")
            df["TRYUSD"] = df[close_col].ffill().bfill()
            df = df.drop(columns=["Date", close_col, "date_only"])

        except Exception as e:
            print(f"Error fetching FX data: {e}")

    else:
        print("No valid date values found in df.")

    print(f"\n{fx_ticker} feature added.")
    return df


def add_holidays(df, date_column, urun_column, sales_column):
    """Adds a holiday marker column based on Turkish holidays."""
    # date_column = "SAS_Date"
    # urun_column = "URUN_KODU"
    # sales_column = "ACTUAL"
    print(f"\nCalculating holiday feature based on {date_column}...")

    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")
    years = df[date_column].dropna().dt.year.unique()
    turkey_holidays = set()

    for year in years:
        turkey_holidays.update(holidays.Turkey(years=int(year)).keys())

    df["holiday_marker"] = df[date_column].dt.date.apply(lambda x: 1 if x in turkey_holidays else 0)

    mean_by_urun = df.groupby([urun_column, "holiday_marker"])[sales_column].mean().unstack(fill_value=0)
    mean_by_urun.columns = (
        ["Non-Holidays Mean Sales", "Holidays Mean Sales"] if 0 in mean_by_urun.columns else mean_by_urun.columns
    )
    print(mean_by_urun)
    print("\nHoliday feature added.")
    return df


def time_series_decomposition(df, key_column, date_column, sales_column, min_data_points, seasonal_period):
    # This can be slow when there are many series.
    """Adds trend, seasonal, and residual components to each series using seasonal decomposition."""
    # date_column = "SAS_Date"
    # urun_column = "URUN_KODU"
    # sales_column = "ACTUAL"
    # min_data_points = 12
    # seasonal_period = 3
    print(f"\nCalculating time series decomposition for {sales_column}...")

    df[date_column] = pd.to_datetime(df[date_column])
    df = df.sort_values([key_column, date_column])
    df["trend"] = None
    df["seasonal"] = None
    df["residual"] = None

    for key, group in df.groupby(key_column):
        ts = group.set_index(date_column)[sales_column]

        if len(ts) < min_data_points:
            print(f"Skipping {key}: not enough data points ({len(ts)})")
            continue
        result = seasonal_decompose(ts, model="additive", period=seasonal_period)

        df.loc[group.index, "trend"] = result.trend.values
        df.loc[group.index, "seasonal"] = result.seasonal.values
        df.loc[group.index, "residual"] = result.resid.values

    print("\nTime series decomposition completed.")
    return df


df = lagged_sales(a, key_column="unq_key", date_column="month", sales_column="actual_m", fill_na=True, max_lag=10)
df = pct_change_sales(df, key_column="unq_key", date_column="month", sales_column="actual_m", fill_na=True, max_lag=10)
df = rolling_avg_sales(df, key_column="unq_key", date_column="month", sales_column="actual_m", window_size=10)
df = add_tryusd(df, date_column="month", fx_ticker="TRYUSD=X")

In [ ]:
import pandas as pd
import numpy as np
def load_prep_data(df) -> pd.DataFrame:
    """Load and preprocess feature data from DB before prediction month.
    Reads table FEATURE_TABLE and filters records up to pred_month.
    """
    df["pct_change_sales"] = df["pct_change_sales"].clip(lower=0, upper=1)

    cols_to_fill = [
        "actual_m", "lagged_sales", "pct_change_sales", "rolling_avg_sales",
        "TRYUSD", "ratio_in_channel_m", "ratio_in_cat_m",
        "mom_growth"
    ]

    # --- Normalize month to [0, 1] ---
    df["month_"] = (df["month"].dt.month - 1) / 11

    # --- Handle missing values efficiently ---
    df = df.sort_values(["unq_key", "month"])
    df[cols_to_fill] = df[cols_to_fill].replace([np.inf, -np.inf], np.nan)
    df[cols_to_fill] = df[cols_to_fill].bfill().ffill().fillna(0)

    # --- Aggregate time-series into lists per key ---
    final_df = (
        df.groupby("unq_key", as_index=False)
          .agg({
              "month_": list,
              "actual_m": list,
              "lagged_sales": list,
              "pct_change_sales": list,
              "rolling_avg_sales": list,
              "TRYUSD": list,
              "ratio_in_channel_m": list,
              "ratio_in_cat_m": list,
              "mom_growth": list,
              "yoy_growth": list,
              "month_": list
          })
    )

    # --- Helper functions ---
    def lag_list_fillna(lst, fill_first=0):
        """
        Shift list by one and fill first missing with given value.
        """
        if not isinstance(lst, list) or len(lst) == 0:
            return []

        lagged = [np.nan] + lst[:-1]
        filled = []
        for i, x in enumerate(lagged):
            if pd.isnull(x):
                if i > 0 and not pd.isnull(filled[i - 1]):
                    filled.append(filled[i - 1])
                else:
                    filled.append(fill_first)
            else:
                filled.append(x)
        return filled

    def log_list(lst):
        """
        Elementwise ln transformation for lists.
        """
        if not isinstance(lst, list):
            return []
        result = []
        for x in lst:
            if pd.isnull(x):
                result.append(0)
            elif x < 0:
                result.append(-np.log1p(abs(x)))  # preserve sign
            elif abs(x) > 0.001:
                result.append(np.log1p(x))
            else:
                result.append(0)
        return result

    # --- Apply lag fill for regular numeric columns ---
    lag_cols = [
        "actual_m", "lagged_sales", "pct_change_sales", "rolling_avg_sales",
        "TRYUSD", "ratio_in_channel_m", "ratio_in_cat_m", "yoy_growth", "mom_growth"
    ]

    for col in lag_cols:
        final_df[f"{col}_lag1"] = final_df[col].apply(lag_list_fillna)

    # --- Month lag uses different fill value (1 instead of 0) ---
    final_df["month_lag1"] = final_df["month_"].apply(lambda x: lag_list_fillna(x, fill_first=1))

    # --- Apply log transforms ---
    log_targets = [
        "actual_m", "actual_m_lag1",
        "lagged_sales_lag1", "rolling_avg_sales_lag1", "TRYUSD_lag1"
    ]

    for col in log_targets:
        final_df[f"{col}_ln"] = final_df[col].apply(log_list)

    return final_df

df=load_prep_data(df)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# --- CONFIG: list-style features per key ---
key_col    = "unq_key"
target_col = "actual_m"   # list of target values per key

feature_cols = [
    "actual_m_lag1_ln",
    "lagged_sales_lag1_ln",
    "pct_change_sales_lag1",
    "rolling_avg_sales_lag1_ln",
    "TRYUSD_lag1_ln",
    "ratio_in_channel_m_lag1",
    "ratio_in_cat_m_lag1",
    "month_lag1",
]

# --- KNN tuning per key (time-series within each row) ---
param_grid = [(3, "uniform"), (3, "distance"), (5, "uniform"), (5, "distance")]
results = []

for _, row in df.iterrows():
    key = row[key_col]

    # y: target time-series
    series_y = row[target_col]
    if not isinstance(series_y, list):
        continue
    y = np.asarray(series_y, dtype=float)
    n = len(y)

    # Skip keys without enough time steps
    if n < 6:
        continue

    # X: stack feature time-series columns -> shape (n, n_features)
    try:
        X = np.column_stack([row[col] for col in feature_cols])
    except KeyError as e:
        # Missing feature column for this row/key
        continue

    # Safety check: feature length must match target length
    if X.shape[0] != n:
        continue

    # Use last 3 points as validation, others as train
    split = n - 3
    X_train, y_train = X[:split], y[:split]
    X_val, y_val   = X[split:], y[split:]

    best_score = np.inf
    best_k = None
    best_w = None

    for k, w in param_grid:
        if split < k:
            # not enough train samples for this k
            continue
        model = KNeighborsRegressor(n_neighbors=k, weights=w)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        mse = mean_squared_error(y_val, y_pred)

        if mse < best_score:
            best_score = mse
            best_k = k
            best_w = w

    if best_k is not None:
        results.append({
            key_col: key,
            "best_k": best_k,
            "best_distance": best_w,
            "best_mse": best_score,
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
import os

out_path = os.getenv("KNN_TUNE_OUTPUT_PATH", "knn_tune_results.xlsx")
results_df.to_excel(out_path, index=False)

In [ ]:
import os
from sklearn.feature_selection import SequentialFeatureSelector
from joblib import Parallel, delayed

if 'results_df' not in globals():
    raise RuntimeError("results_df not found. Run the KNN tuning cell first.")

key_col      = 'unq_key'
target_col   = 'actual_m'
feature_cols = [
    "actual_m_lag1_ln",
    "lagged_sales_lag1_ln",
    "pct_change_sales_lag1",
    "rolling_avg_sales_lag1_ln",
    "TRYUSD_lag1_ln",
    "ratio_in_channel_m_lag1",
    "ratio_in_cat_m_lag1",
    "month_lag1",
]

# ── Pre-index df by key to avoid O(n) scan per key ──────────────────────────
df_indexed = df.set_index(key_col)

def process_key(res):
    key    = res[key_col]
    best_k = int(res["best_k"])
    best_w = res["best_distance"]

    if key not in df_indexed.index:
        return None

    row    = df_indexed.loc[key]
    series_y = row[target_col]
    if not isinstance(series_y, list):
        return None

    y = np.asarray(series_y, dtype=float)
    n = len(y)
    if n < 6:
        return None

    # ── Validate & collect feature arrays in one pass ───────────────────────
    feature_arrays = []
    for col in feature_cols:
        series_x = row.get(col)
        if not isinstance(series_x, list) or len(series_x) != n:
            return None
        feature_arrays.append(np.asarray(series_x, dtype=float))

    # ── Vectorised X construction (no Python loop over t) ───────────────────
    X = np.column_stack(feature_arrays)   # shape (n, n_features)

    # ── n_jobs=1 here — parallelism is at the key level ─────────────────────
    base_estimator = KNeighborsRegressor(n_neighbors=best_k, weights=best_w)
    sfs = SequentialFeatureSelector(
        base_estimator,
        n_features_to_select="auto",
        direction="forward",
        scoring="neg_mean_squared_error",
        n_jobs=1,   # ← critical: avoids deadlock with outer Parallel
        tol=None,
    )

    try:
        sfs.fit(X, y)
    except Exception as e:
        print(f"Skipping key {key} during SFS: {e}")
        return None

    selected_features = [f for f, m in zip(feature_cols, sfs.get_support()) if m]
    return {
        key_col: key,
        "best_k": best_k,
        "best_distance": best_w,
        "selected_features": selected_features,
    }

# ── Run all keys in parallel ─────────────────────────────────────────────────
rows = results_df.to_dict("records")   # avoids iterrows() overhead

sfs_rows = Parallel(n_jobs=4, backend="threading", verbose=5)(
    delayed(process_key)(res) for res in rows
)

sfs_results_df = pd.DataFrame([r for r in sfs_rows if r is not None])
print("SFS done. Keys processed:", len(sfs_results_df))
sfs_results_df.head()

out_path = os.getenv("KNN_SFS_OUTPUT_PATH", "knn_sfs_results.xlsx")
sfs_results_df.to_excel(out_path, index=False)